# CycloSafe - Cyclone Detection Model Training (V3)
## Objective 1: Binary Classification (Cyclone vs No_Cyclone)

**Dataset:** 5.48 Lakh NASA GIBS Satellite Images

**Architecture:** EfficientNetB0 (Transfer Learning from ImageNet)

## Step 1: Mount Google Drive and Unzip Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import zipfile, os

PARTS = [
    '/content/drive/MyDrive/dataset_part_1.zip',
    '/content/drive/MyDrive/dataset_part_2.zip',
    '/content/drive/MyDrive/dataset_part_3.zip',
    '/content/drive/MyDrive/dataset_part_4.zip',
]
EXTRACT_DIR = '/content/dataset_v2'

if not os.path.exists(os.path.join(EXTRACT_DIR, 'Cyclone')):
    for i, part in enumerate(PARTS, 1):
        print(f'Extracting Part {i}/4: {part}...')
        with zipfile.ZipFile(part, 'r') as zf:
            zf.extractall(EXTRACT_DIR)
    print('All parts extracted!')
else:
    print('Dataset already extracted.')

c_count = len(os.listdir(os.path.join(EXTRACT_DIR, 'Cyclone')))
nc_count = len(os.listdir(os.path.join(EXTRACT_DIR, 'No_Cyclone')))
print(f'Cyclone: {c_count:,} | No_Cyclone: {nc_count:,} | Total: {c_count+nc_count:,}')


## Step 2: Import Libraries and Set Hyperparameters

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import seaborn as sns
import json, time, shutil, random

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

IMG_SIZE = (224, 224)
BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE = 1e-4
SEED = 42


## Step 3: Create Train / Validation / Test Split

In [ ]:
SPLIT_DIR = '/content/dataset_split'

def create_splits(src_dir, split_dir, val_ratio=0.15, test_ratio=0.10):
    if os.path.exists(os.path.join(split_dir, 'train', 'Cyclone')):
        print('Splits already exist.')
        return
    for split in ['train', 'val', 'test']:
        for cls in ['Cyclone', 'No_Cyclone']:
            os.makedirs(os.path.join(split_dir, split, cls), exist_ok=True)
    for cls in ['Cyclone', 'No_Cyclone']:
        cls_dir = os.path.join(src_dir, cls)
        files = [f for f in os.listdir(cls_dir) if f.endswith(('.jpg','.jpeg','.png'))]
        random.seed(SEED)
        random.shuffle(files)
        n = len(files)
        n_test = int(n * test_ratio)
        n_val = int(n * val_ratio)
        n_train = n - n_test - n_val
        splits = {'train': files[:n_train], 'val': files[n_train:n_train+n_val], 'test': files[n_train+n_val:]}
        for sname, sfiles in splits.items():
            dest = os.path.join(split_dir, sname, cls)
            for f in sfiles:
                os.symlink(os.path.join(cls_dir, f), os.path.join(dest, f))
            print(f'  {cls}/{sname}: {len(sfiles):,}')

print('Creating splits...')
create_splits(EXTRACT_DIR, SPLIT_DIR)

for split in ['train', 'val', 'test']:
    c = len(os.listdir(os.path.join(SPLIT_DIR, split, 'Cyclone')))
    nc = len(os.listdir(os.path.join(SPLIT_DIR, split, 'No_Cyclone')))
    print(f'{split.upper()}: {c+nc:,} (Cyclone: {c:,} | No_Cyclone: {nc:,})')


## Step 4: Build Data Generators with Augmentation

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0, rotation_range=30, width_shift_range=0.15,
    height_shift_range=0.15, shear_range=0.1, zoom_range=0.2,
    horizontal_flip=True, vertical_flip=True, brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)
val_test_datagen = ImageDataGenerator(rescale=1.0/255.0)

train_gen = train_datagen.flow_from_directory(os.path.join(SPLIT_DIR,'train'), target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=True, seed=SEED)
val_gen = val_test_datagen.flow_from_directory(os.path.join(SPLIT_DIR,'val'), target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=False)
test_gen = val_test_datagen.flow_from_directory(os.path.join(SPLIT_DIR,'test'), target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=False)

print(f'Classes: {train_gen.class_indices}')
print(f'Train batches: {len(train_gen)} | Val batches: {len(val_gen)} | Test batches: {len(test_gen)}')


## Step 5: Visualize Sample Augmented Images

In [ ]:
batch_imgs, batch_labels = next(train_gen)
class_names = list(train_gen.class_indices.keys())
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
fig.suptitle('Sample Augmented Training Images', fontsize=16, fontweight='bold')
for i, ax in enumerate(axes.flat):
    if i < len(batch_imgs):
        ax.imshow(batch_imgs[i])
        label = class_names[int(batch_labels[i])]
        ax.set_title(label, color='red' if label=='Cyclone' else 'green', fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.show()


## Step 6: Build EfficientNetB0 Model

In [ ]:
def build_detection_model(input_shape=(224, 224, 3)):
    aug_model = keras.Sequential([
        layers.RandomFlip('horizontal_and_vertical'),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.1),
        layers.RandomContrast(0.1),
    ], name='augmentation')
    base_model = tf.keras.applications.EfficientNetB0(weights='imagenet', include_top=False, input_shape=input_shape)
    base_model.trainable = False
    inputs = layers.Input(shape=input_shape, name='satellite_input')
    x = aug_model(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu', name='dense_256')(x)
    x = layers.Dropout(0.4, name='dropout_04')(x)
    x = layers.Dense(128, activation='relu', name='dense_128')(x)
    x = layers.Dropout(0.3, name='dropout_03')(x)
    outputs = layers.Dense(1, activation='sigmoid', name='cyclone_probability')(x)
    model = models.Model(inputs=inputs, outputs=outputs, name='CycloSafe_Detector_V3')
    return model, base_model

model, base_model = build_detection_model()
model.summary()


## Step 7: Phase 1 Training - Feature Extraction (Frozen Base)

In [ ]:
model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

phase1_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1),
]

print('PHASE 1: Feature Extraction (Base Model Frozen)')
t0 = time.time()
history_phase1 = model.fit(train_gen, epochs=10, validation_data=val_gen, callbacks=phase1_callbacks, verbose=1)
phase1_time = time.time() - t0
print(f'Phase 1 completed in {phase1_time/60:.1f} minutes')


## Step 8: Phase 2 Training - Fine-Tuning (Unfrozen Base)

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(optimizer=optimizers.Adam(learning_rate=LEARNING_RATE), loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

phase2_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1),
    callbacks.ModelCheckpoint('/content/best_cyclone_detector.keras', monitor='val_accuracy', save_best_only=True, verbose=1),
]

print('PHASE 2: Fine-Tuning (Base Model Partially Unfrozen)')
t0 = time.time()
history_phase2 = model.fit(train_gen, epochs=EPOCHS, validation_data=val_gen, callbacks=phase2_callbacks, verbose=1)
phase2_time = time.time() - t0
print(f'Phase 2 completed in {phase2_time/60:.1f} minutes')
print(f'Total training time: {(phase1_time + phase2_time)/60:.1f} minutes')


## Step 9: Plot Training History

In [ ]:
def combine_histories(h1, h2):
    combined = {}
    for key in h1.history:
        combined[key] = h1.history[key] + h2.history[key]
    return combined

full_history = combine_histories(history_phase1, history_phase2)
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('CycloSafe Detection Model - Training History', fontsize=16, fontweight='bold')
for i, (metric, label) in enumerate([('accuracy','Accuracy'),('loss','Loss'),('auc','AUC')]):
    axes[i].plot(full_history[metric], label=f'Train {label}', linewidth=2)
    axes[i].plot(full_history[f'val_{metric}'], label=f'Val {label}', linewidth=2)
    axes[i].set_title(label, fontsize=14)
    axes[i].set_xlabel('Epoch')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/training_history.png', dpi=150, bbox_inches='tight')
plt.show()


## Step 10: Evaluate on Test Set

In [ ]:
best_model = keras.models.load_model('/content/best_cyclone_detector.keras')
test_loss, test_accuracy, test_auc = best_model.evaluate(test_gen, verbose=1)
print(f'Test Accuracy: {test_accuracy*100:.2f}% | Test AUC: {test_auc:.4f}')

test_gen.reset()
y_pred_prob = best_model.predict(test_gen, verbose=1)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()
y_true = test_gen.classes
class_names = list(test_gen.class_indices.keys())

print('\nCLASSIFICATION REPORT')
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
roc_score = roc_auc_score(y_true, y_pred_prob)
print(f'ROC-AUC Score: {roc_score:.4f}')


## Step 11: Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_title('Confusion Matrix - CycloSafe Detection V3', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'TP: {tp:,} | TN: {tn:,} | FP: {fp:,} | FN: {fn:,}')
print(f'Precision: {tp/(tp+fp)*100:.2f}% | Recall: {tp/(tp+fn)*100:.2f}% | F1: {2*tp/(2*tp+fp+fn)*100:.2f}%')


## Step 12: Find Optimal Threshold and Save Config

In [ ]:
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_true, y_pred_prob)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = float(thresholds[best_idx])
print(f'Optimal Threshold: {best_threshold:.6f} | Best F1: {f1_scores[best_idx]:.4f}')

config = {'threshold': best_threshold, 'test_accuracy': float(test_accuracy), 'test_auc': float(test_auc), 'roc_auc': float(roc_score), 'class_names': class_names, 'total_training_images': c_count + nc_count, 'model_version': 'V3'}
with open('/content/detection_v3_config.json', 'w') as f:
    json.dump(config, f, indent=4)
print(json.dumps(config, indent=4))


## Step 13: Save Model to Google Drive

In [ ]:
SAVE_DIR = '/content/drive/MyDrive/CycloSafe_Models'
os.makedirs(SAVE_DIR, exist_ok=True)
shutil.copy('/content/best_cyclone_detector.keras', os.path.join(SAVE_DIR, 'cyclone_detector.keras'))
shutil.copy('/content/detection_v3_config.json', os.path.join(SAVE_DIR, 'detection_v3_config.json'))
shutil.copy('/content/training_history.png', os.path.join(SAVE_DIR, 'training_history.png'))
shutil.copy('/content/confusion_matrix.png', os.path.join(SAVE_DIR, 'confusion_matrix.png'))
print('MODEL SAVED TO GOOGLE DRIVE!')
print(f'Location: {SAVE_DIR}')


## Step 14: Download Model Directly

In [ ]:
from google.colab import files
files.download('/content/best_cyclone_detector.keras')
files.download('/content/detection_v3_config.json')


---
## Training Complete!

**Next Steps:**
1. Download `cyclone_detector.keras` and `detection_v3_config.json` from Google Drive
2. Place them in `backend/models/`
3. Restart your backend server
4. Your dashboard will now use the new V3 model trained on 5.48 Lakh images!

---
*CycloSafe - AI-Based Tropical Cyclone Monitoring System*